# MNIST Autoencoder for Image Denoising

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data.sampler import SubsetRandomSampler

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

### Loading the dataset

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=transform)

num_train = len(train_data)
indices = list(range(num_train))
np.random.shuffle(indices)

valid_size = 12000
train_idx, valid_idx = indices[valid_size:], indices[:valid_size]

train_sampler = SubsetRandomSampler(train_idx)
valid_sampler = SubsetRandomSampler(valid_idx)

batch_size = 64

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=train_sampler)
valid_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=valid_sampler)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False)

print(len(train_idx), len(valid_idx), len(test_data))

100%|██████████| 9.91M/9.91M [00:05<00:00, 1.85MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 94.9kB/s]


### Visualizing a batch

In [ ]:
dataiter = iter(train_loader)
images, labels = next(dataiter)

fig = plt.figure(figsize=(12,3))
for i in range(10):
    ax = fig.add_subplot(1, 10, i+1)
    ax.imshow(images[i].numpy().squeeze(), cmap='gray')
    ax.axis('off')

### Adding noise
We'll be feeding noisy images to the network and training it to reconstruct the original, clean image.

In [ ]:
noise_factor = 0.4

def add_noise(img, factor=noise_factor):
    noisy_img = img + factor*torch.randn(*img.shape)
    noisy_img = torch.clip(noisy_img, 0., 1.)
    return noisy_img

In [ ]:
noisy_images = add_noise(images)

fig = plt.figure(figsize=(12,3))
for i in range(10):
    ax = fig.add_subplot(1, 10, i+1)
    ax.imshow(noisy_images[i].numpy().squeeze(), cmap='gray')
    ax.axis('off')

### Building the network
A convolutional autoencoder. The encoder compresses the 28x28 image down through conv + maxpool layers, and the decoder upsamples it back using transpose convolutions.

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 8, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.t_conv1 = nn.ConvTranspose2d(8, 8, 3, stride=2)
        self.t_conv2 = nn.ConvTranspose2d(8, 16, 2, stride=2)
        self.t_conv3 = nn.ConvTranspose2d(16, 32, 2, stride=2)
        self.conv_out = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)

        x = F.relu(self.t_conv1(x))
        x = F.relu(self.t_conv2(x))
        x = F.relu(self.t_conv3(x))
        x = torch.sigmoid(self.conv_out(x))
        return x

### Weight initialization
Weights for the conv layers are sampled from a normal distribution, biases start at 0.

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        y = m.in_channels
        m.weight.data.normal_(0.0, 1/np.sqrt(y))
        m.bias.data.fill_(0)

In [ ]:
model = ConvAutoencoder()
model.apply(init_weights)
model.to(device)
model

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Training

In [ ]:
epochs = 20
valid_loss_min = np.inf

train_losses, valid_losses = [], []

for epoch in range(1, epochs+1):
    train_loss = 0.0
    valid_loss = 0.0

    model.train()
    for images, _ in train_loader:
        noisy_imgs = add_noise(images).to(device)
        images = images.to(device)

        optimizer.zero_grad()
        outputs = model(noisy_imgs)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()*images.size(0)

    model.eval()
    with torch.no_grad():
        for images, _ in valid_loader:
            noisy_imgs = add_noise(images).to(device)
            images = images.to(device)

            outputs = model(noisy_imgs)
            loss = criterion(outputs, images)
            valid_loss += loss.item()*images.size(0)

    train_loss = train_loss/len(train_idx)
    valid_loss = valid_loss/len(valid_idx)
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'Epoch {epoch}/{epochs} \tTraining Loss: {train_loss:.6f} \tValidation Loss: {valid_loss:.6f}')

    if valid_loss <= valid_loss_min:
        torch.save(model.state_dict(), 'mnist_denoising_model.pt')
        valid_loss_min = valid_loss

### Loss curves

In [ ]:
plt.plot(train_losses, label='Training loss')
plt.plot(valid_losses, label='Validation loss')
plt.xlabel('epoch')
plt.ylabel('MSE loss')
plt.legend()
plt.show()

### Testing on the test set

In [ ]:
model.load_state_dict(torch.load('mnist_denoising_model.pt'))
model.eval()

test_loss = 0.0
with torch.no_grad():
    for images, _ in test_loader:
        noisy_imgs = add_noise(images).to(device)
        images = images.to(device)

        outputs = model(noisy_imgs)
        loss = criterion(outputs, images)
        test_loss += loss.item()*images.size(0)

test_loss = test_loss/len(test_data)
print(f'Test Loss: {test_loss:.6f}')

### Denoised output

In [ ]:
dataiter = iter(test_loader)
images, labels = next(dataiter)
noisy_imgs = add_noise(images)

with torch.no_grad():
    output = model(noisy_imgs.to(device))
output = output.cpu().numpy()

fig, axes = plt.subplots(nrows=3, ncols=10, figsize=(15,4))

for imgs, row in zip([images, noisy_imgs, output], axes):
    for img, ax in zip(imgs, row):
        ax.imshow(np.squeeze(img), cmap='gray')
        ax.axis('off')

axes[0,0].set_title('original')
axes[1,0].set_title('noisy')
axes[2,0].set_title('denoised')
plt.tight_layout()
plt.show()